# Inhouse TTS · entrenamiento del modelo propio (IHV1) en Google Colab

Este cuaderno entrena **nuestro propio modelo de voz**, desde cero y sin modelos de terceros, en la GPU de Colab. Todo el código que entrena es **Rust** (framework *burn* con CUDA); el cuaderno solo lanza los comandos.

**Cómo usarlo**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU*** (T4 en la versión gratuita; L4 o A100 en Colab Pro).
2. *Entorno de ejecución → **Ejecutar todo***. La primera vez, Colab pide permiso para usar tu Google Drive.
3. Deja la pestaña abierta mientras entrena. Si la sesión se corta, vuelve a abrir el cuaderno y pulsa *Ejecutar todo*: el entrenamiento **continúa desde el último checkpoint** (cada 1000 pasos, en `MyDrive/inhouse-tts/runs/<voz>`).

| Paso | Tiempo |
|---|---|
| Entrenador en Rust | ~1 min (binario precompilado) |
| Datos | ~2 min (corpus ya preparado, publicado en GitHub) |
| Entrenar | las horas que dure la sesión; continúa en la siguiente |

`MyDrive/inhouse-tts/runs/<voz>/samples/` guarda muestras de audio cada 2000 pasos para escuchar el progreso. En Drive solo se guardan los checkpoints (~1 GB), no los datos.

**Preset**: `auto` usa `base` (el modelo grande, más natural) en GPUs de 24 GB o más (A100, L4) y `small` en la T4. El preset queda fijado al empezar; para cambiarlo, borra `MyDrive/inhouse-tts/runs/<voz>`.

**Datos de `es-ES`** (castellano con distinción, banda completa): lectora de LibriVox *Mongope* (dominio público), cortada de los MP3 originales de 128 kbps con el índice de CML-TTS (CC BY 4.0), y el conjunto *dave* de Open Home Foundation (CC0).

In [ ]:
#@title Configuración
VOICE = "es-ES"   #@param ["es-ES"]
PRESET = "auto"   #@param ["auto", "base", "small", "tiny"]
STEPS = 300000    #@param {type:"integer"}
REPO = "https://github.com/miguelcoxcaballero/inhouse-TTS"
import os
WORK = "/content/drive/MyDrive/inhouse-tts"
os.environ.update(VOICE=VOICE, PRESET=PRESET, STEPS=str(STEPS), REPO=REPO, WORK=WORK)

In [ ]:
#@title 1 · GPU y Google Drive
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "$WORK/runs"

### 2 · Entrenador en Rust: binario precompilado (CUDA) o compilación desde el código

In [ ]:
%%bash
set -e
cd /content
URL="$REPO/releases/download/trainer-latest/ihtts-trainer-cuda-linux-x86_64.tar.gz"
if curl -fsSL "$URL" -o trainer.tgz && tar -xzf trainer.tgz; then
  REV=$(cat ihtts/REVISION)
  echo "binario precompilado de $REV"
else
  REV=main
fi
test -d inhouse-tts || git clone -q "$REPO" inhouse-tts
cd inhouse-tts && git fetch -q origin && git checkout -q "$REV" && cd ..
mkdir -p bin
if [ -x ihtts/ihtts-train ]; then
  cp ihtts/ihtts-train ihtts/ihtts-data bin/
else
  test -x $HOME/.cargo/bin/cargo || (curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q)
  export PATH=$HOME/.cargo/bin:$PATH
  cd inhouse-tts
  cargo build --profile train -p inhouse-tts-data -p inhouse-tts-train --features inhouse-tts-train/cuda
  cp target/train/ihtts-train target/train/ihtts-data ../bin/
fi
/content/bin/ihtts-train 2>&1 | head -1

### 3 · Datos: corpus preparado (release `data-<voz>` en GitHub) o preparación aquí

In [ ]:
%%bash
set -e
cd /content/inhouse-tts
mkdir -p /content/data
if [ -f /content/data/.ready ]; then
  echo "datos ya descargados"
elif curl -fsSL "$REPO/releases/download/data-$VOICE/data-$VOICE.parts" -o /content/parts.txt; then
  for p in $(cat /content/parts.txt); do
    curl -fsSL "$REPO/releases/download/data-$VOICE/$p"
  done | tar -x -C /content/data
  touch /content/data/.ready
else
  echo "no hay corpus publicado; se prepara aquí (puede tardar una hora)"
  /content/bin/ihtts-data prepare corpora/$VOICE/*.json --cache /content/cache --out /content/data
  rm -rf /content/cache
  touch /content/data/.ready
fi
for r in /content/data/*/report.json; do
  echo "$(dirname $r | xargs basename): $(grep -E '"(utterances|hours)"' $r | tr -d ' \n')"
done

### 4 · Entrenar (continúa automáticamente desde el último checkpoint)

In [ ]:
%%bash
cd /content/inhouse-tts
# Bibliotecas CUDA de Colab (el entrenador las carga al arrancar).
export LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/cuda/lib64:${LD_LIBRARY_PATH}
# auto: "base" (el modelo grande) con ≥ 24 GB de GPU (A100, L4), "small" en la T4.
if [ "$PRESET" = auto ]; then
  MEM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1)
  if [ "${MEM:-0}" -ge 22000 ]; then PRESET=base; else PRESET=small; fi
fi
if [ -f "$WORK/runs/$VOICE/config.json" ]; then
  echo "continuando el entrenamiento existente ($(grep -o '"ihv1-[a-z]*"' "$WORK/runs/$VOICE/config.json"))"
else
  echo "nuevo entrenamiento con el preset $PRESET"
fi
DATA=$(ls /content/data/*/manifest.jsonl | sed 's/^/--data /' | tr '\n' ' ')
/content/bin/ihtts-train train --run "$WORK/runs/$VOICE" --preset "$PRESET" $DATA \
  --device cuda --steps "$STEPS" --resume 2>&1

In [ ]:
#@title 5 · Escuchar las últimas muestras
import glob, IPython.display as ipd
latest = sorted(glob.glob(f"{WORK}/runs/{VOICE}/samples/*"))[-1:]
for d in latest:
    print(d)
    for w in sorted(glob.glob(d + "/*.wav"))[:6]:
        print(os.path.basename(w)); ipd.display(ipd.Audio(w))

In [ ]:
#@title 6 · Exportar el modelo al formato de Inhouse TTS (.ihw)
!/content/bin/ihtts-train export --run "$WORK/runs/$VOICE" --out "$WORK/ihv1-$VOICE.ihw"
print("Modelo en Google Drive:", f"{WORK}/ihv1-{VOICE}.ihw")